In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
traindf = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
testdf = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
print(traindf.shape, testdf.shape)

In [ ]:
traindf.info()

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb

# ------------------------
# 📦 1. Load & Preprocess Data
# ------------------------

# Drop columns with >= 50% nulls
traindf.dropna(axis=1, thresh=int(0.5 * len(traindf)), inplace=True)
testdf.dropna(axis=1, thresh=int(0.5 * len(testdf)), inplace=True)

# Align columns in both train and test
common_cols = traindf.columns.intersection(testdf.columns)
common_cols = common_cols.drop('SalePrice', errors='ignore')  # Drop if it exists
traindf = traindf[common_cols.tolist() + ['SalePrice']]
testdf = testdf[common_cols]

# Separate target and features
y = traindf['SalePrice']
X = traindf.drop(['SalePrice'], axis=1)

# ------------------------
# 🔍 2. Split Data
# ------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.95, test_size=0.05, random_state=0
)

# ------------------------
# 🛠️ 3. Define Preprocessing Pipeline
# ------------------------

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# Numerical pipeline
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Full preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ]
)

# ------------------------
# 🤖 4. Define Models
# ------------------------

models = {
    "LinearReg": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=0),
    "LightGBM": lgb.LGBMRegressor(random_state=0),
    "XGBoost": xgb.XGBRegressor(random_state=0)
}

results = {}

# ------------------------
# 🧪 5. Train and Evaluate
# ------------------------

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    y_preds = pipe.predict(X_valid)

    mse = mean_squared_error(y_valid, y_preds)
    r2 = r2_score(y_valid, y_preds)

    results[name] = {'MSE': mse, 'R2': r2}
    print(f"{name} - MSE: {mse:.2f}, R2: {r2:.4f}")

# ------------------------
# 🏆 6. Final Prediction with Best Model
# ------------------------

# Pick best model (Random Forest here, or based on best R2)
best_model_name = max(results, key=lambda k: results[k]['R2'])
best_model = models[best_model_name]

final_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_model)
])

final_pipe.fit(X, y)
test_predictions = final_pipe.predict(testdf)

# ------------------------
# 📤 7. Save Submission
# ------------------------

submission = pd.DataFrame({
    'Id': testdf["Id"],
    'SalePrice': test_predictions
})

submission.to_csv('submission.csv', index=False)
print(f"✅ Submission file 'submission.csv' created using {best_model_name}!")
